# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/hanisbev/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 21:02:17.518973: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 21:02:17.521482: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 21:02:17.556947: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-20 21:02:17.556987: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-20 21:02:17.557021: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
# Feature matrix (all columns except "Class")
X = df.drop(columns=["Class"])

# Class labels
y = df["Class"]



In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# <-- Enter your code here <--#

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()

# Fit the scaler on the training data
# and transform both training and test sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Check results
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

# Preview first scaled sample
print("\nFirst scaled training sample:\n", X_train_scaled[0])

X_train_scaled shape: (124, 13)
X_test_scaled shape: (54, 13)

First scaled training sample:
 [ 0.62844732  1.08120605 -0.65212742  0.         -0.8414766  -1.00335756
 -1.51706225  1.71144809 -1.23077056  0.33317435 -0.64137827 -1.07090115
 -0.51821917]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_encoded = tf.keras.utils.to_categorical(
    y_train,
    num_classes=num_classes
)

y_test_encoded = tf.keras.utils.to_categorical(
    y_test,
    num_classes=num_classes
)

# Check shapes
print("y_train_encoded shape:", y_train_encoded.shape)
print("y_test_encoded shape:", y_test_encoded.shape)


y_train_encoded shape: (124, 3)
y_test_encoded shape: (54, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

2026-05-20 21:02:21.451557: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-20 21:02:21.452286: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train,
    y_train_encoded,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 1s 20ms/step - loss: 38.4984 - accuracy: 0.3434 - val_loss: 13.0618 - val_accuracy: 0.2800
Epoch 2/20
13/13 [==============================] - 0s 6ms/step - loss: 9.3350 - accuracy: 0.4343 - val_loss: 13.8834 - val_accuracy: 0.2800
Epoch 3/20
13/13 [==============================] - 0s 5ms/step - loss: 6.9624 - accuracy: 0.4646 - val_loss: 5.3311 - val_accuracy: 0.5200
Epoch 4/20
13/13 [==============================] - 0s 4ms/step - loss: 3.2669 - accuracy: 0.6364 - val_loss: 3.4346 - val_accuracy: 0.5600
Epoch 5/20
13/13 [==============================] - 0s 5ms/step - loss: 3.0786 - accuracy: 0.6162 - val_loss: 3.2904 - val_accuracy: 0.6400
Epoch 6/20
13/13 [==============================] - 0s 4ms/step - loss: 2.1431 - accuracy: 0.6768 - val_loss: 3.1068 - val_accuracy: 0.6000
Epoch 7/20
13/13 [==============================] - 0s 4ms/step - loss: 4.5289 - accuracy: 0.5758 - val_loss: 4.4122 - val_accuracy: 0.6000
Epoch 8/20
13/13

In [12]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)
y_true = y_test_encoded.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred_classes))
print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))


2/2 [==============================] - 0s 4ms/step
Accuracy: 0.7592592592592593
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       1.00      0.43      0.60        21
           2       0.52      1.00      0.68        14

    accuracy                           0.76        54
   macro avg       0.84      0.79      0.75        54
weighted avg       0.88      0.76      0.75        54

[[18  0  1]
 [ 0  9 12]
 [ 0  0 14]]


In [13]:
import os
# Create TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)


# Convert model
tflite_model = converter.convert()

# Save model
tflite_path = "wine_model.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

# Print file size in KB
size_kb = os.path.getsize(tflite_path) / 1024

print("Saved TFLite model as:", tflite_path)
print(f"Model size: {size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpg56yga3i/assets


INFO:tensorflow:Assets written to: /tmp/tmpg56yga3i/assets
2026-05-20 21:02:24.165102: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:24.165160: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


Saved TFLite model as: wine_model.tflite
Model size: 14.06 KB


2026-05-20 21:02:24.165447: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpg56yga3i
2026-05-20 21:02:24.166827: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:24.166845: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpg56yga3i
2026-05-20 21:02:24.172318: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 21:02:24.173130: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:24.218002: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpg56yga3i
2026-05-20 21:02:24.231122: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 65673 microseconds.
2026-05-20 21:02:24.245495: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR cr

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Provide representative_data_gen(X_train_scaled).
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)        
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        
        # <-- Enter your code here <--#
        #pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Set supported_types to [tf.float16].
        converter.target_spec.supported_types = [tf.float16]
        # <-- Enter your code here <--#
        #pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # <-- Enter your code here <--#
        #pass

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # <-- Enter your code here <--#
    
    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_content=tflite_model)

    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    input_scale, input_zero_point = input_details[0]['quantization']
    
    output_scale, output_zero_point = output_details[0]['quantization']

    y_pred = []

    for i in range(len(X_test)):
        x = X_test[i:i+1].astype(np.float32)

        if input_details[0]['dtype'] == np.int8:
            x = x / input_scale + input_zero_point
            x = np.round(x).astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], x)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details[0]['index'])

        if output_details[0]['dtype'] == np.int8:
            output = (output - output_zero_point) * output_scale

        y_pred.append(np.argmax(output))

    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    file_size_kb = os.path.getsize(filename) / 1024
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb:.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test, y_test_encoded, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test_encoded, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_encoded, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /tmp/tmpimule9nq/assets


INFO:tensorflow:Assets written to: /tmp/tmpimule9nq/assets
/home/hanisbev/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 21:02:24.968768: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:24.968844: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:02:24.969087: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpimule9nq
2026-05-20 21:02:24.970410: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:24.970432: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpimule9nq
2026-05-20 21:02:24.972595: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.



INT8 TFLite model size: 5.73 KB
              precision    recall  f1-score   support

           0       0.50      0.16      0.24        19
           1       0.72      0.62      0.67        21
           2       0.27      0.57      0.36        14

    accuracy                           0.44        54
   macro avg       0.50      0.45      0.42        54
weighted avg       0.53      0.44      0.44        54

[[ 3  1 15]
 [ 1 13  7]
 [ 2  4  8]]
INFO:tensorflow:Assets written to: /tmp/tmpzy9x2ge1/assets


INFO:tensorflow:Assets written to: /tmp/tmpzy9x2ge1/assets
2026-05-20 21:02:25.622181: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:25.622238: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:02:25.622415: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpzy9x2ge1
2026-05-20 21:02:25.623579: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:25.623594: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpzy9x2ge1
2026-05-20 21:02:25.626986: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:25.675051: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpzy9x2ge1
2026-05-20 21:02:25.689138: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.94 KB
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       1.00      0.43      0.60        21
           2       0.52      1.00      0.68        14

    accuracy                           0.76        54
   macro avg       0.84      0.79      0.75        54
weighted avg       0.88      0.76      0.75        54

[[18  0  1]
 [ 0  9 12]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmpqt4glirp/assets


INFO:tensorflow:Assets written to: /tmp/tmpqt4glirp/assets
2026-05-20 21:02:26.244826: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:26.244882: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.



DYNAMIC TFLite model size: 8.16 KB
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       1.00      0.43      0.60        21
           2       0.52      1.00      0.68        14

    accuracy                           0.76        54
   macro avg       0.84      0.79      0.75        54
weighted avg       0.88      0.76      0.75        54

[[18  0  1]
 [ 0  9 12]
 [ 0  0 14]]


2026-05-20 21:02:26.245041: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpqt4glirp
2026-05-20 21:02:26.246261: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:26.246278: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpqt4glirp
2026-05-20 21:02:26.249730: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:26.289779: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpqt4glirp
2026-05-20 21:02:26.300093: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 55052 microseconds.


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity = 0.5,
    final_sparsity = 0.7,
    begin_step = 0,
    end_step = (X_train.shape[0] // 8) * 10
    )

In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()
model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    tfmot.sparsity.keras.prune_low_magnitude(tf.keras.layers.Dense(64, activation='relu')),
    tfmot.sparsity.keras.prune_low_magnitude(tf.keras.layers.Dense(32, activation='relu')),
    tfmot.sparsity.keras.prune_low_magnitude(tf.keras.layers.Dense(3, activation='softmax'))
    ])
# <-- Enter your code here <--#

In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_encoded,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]
)

Epoch 1/10
13/13 [==============================] - 2s 38ms/step - loss: 28.8310 - accuracy: 0.2929 - val_loss: 3.6177 - val_accuracy: 0.5200
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 6.9129 - accuracy: 0.6162 - val_loss: 3.9915 - val_accuracy: 0.6000
Epoch 3/10
13/13 [==============================] - 0s 5ms/step - loss: 2.2659 - accuracy: 0.5657 - val_loss: 1.8819 - val_accuracy: 0.6400
Epoch 4/10
13/13 [==============================] - 0s 5ms/step - loss: 0.8979 - accuracy: 0.7071 - val_loss: 1.1446 - val_accuracy: 0.6400
Epoch 5/10
13/13 [==============================] - 0s 6ms/step - loss: 0.7006 - accuracy: 0.7576 - val_loss: 1.1680 - val_accuracy: 0.6000
Epoch 6/10
13/13 [==============================] - 0s 5ms/step - loss: 0.8607 - accuracy: 0.7172 - val_loss: 1.3903 - val_accuracy: 0.6400
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.6621 - accuracy: 0.7677 - val_loss: 0.8876 - val_accuracy: 0.7200
Epoch 8/10
13/13 [

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_pruned.tflite") / 1024
print(f"TFLite model size: {file_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpi7fpebql/assets


INFO:tensorflow:Assets written to: /tmp/tmpi7fpebql/assets


TFLite model size: 14.09 KB


2026-05-20 21:02:29.732204: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:29.732267: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:02:29.732423: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpi7fpebql
2026-05-20 21:02:29.733013: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:29.733023: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpi7fpebql
2026-05-20 21:02:29.734558: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:29.749986: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpi7fpebql
2026-05-20 21:02:29.755762: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 23338 m

In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = stripped_model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)

print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 3ms/step
              precision    recall  f1-score   support

           0       0.63      1.00      0.78        19
           1       0.40      0.19      0.26        21
           2       0.29      0.29      0.29        14

    accuracy                           0.50        54
   macro avg       0.44      0.49      0.44        54
weighted avg       0.45      0.50      0.45        54

[[19  0  0]
 [ 7  4 10]
 [ 4  6  4]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
teacher_model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])
# <-- Enter your code here <--#
student_model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
y_teacher_soft = teacher_model.predict(X_train_scaled)

4/4 [==============================] - 0s 3ms/step


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
distill_labels = np.concatenate([y_train_encoded, y_teacher_soft], axis=1)
def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    #pass
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    
    alpha = 0.5
    return alpha * hard_loss + (1 - alpha) * soft_loss

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(),
    loss=distillation_loss
)

history = student_model.fit(
    X_train,
    distill_labels,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
)

Epoch 1/10
13/13 [==============================] - 1s 17ms/step - loss: 31.3189 - val_loss: 9.4737
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 8.0953 - val_loss: 5.8376
Epoch 3/10
13/13 [==============================] - 0s 5ms/step - loss: 5.4504 - val_loss: 4.0384
Epoch 4/10
13/13 [==============================] - 0s 4ms/step - loss: 3.7969 - val_loss: 2.9817
Epoch 5/10
13/13 [==============================] - 0s 5ms/step - loss: 2.9479 - val_loss: 1.9845
Epoch 6/10
13/13 [==============================] - 0s 5ms/step - loss: 2.2226 - val_loss: 1.8745
Epoch 7/10
13/13 [==============================] - 0s 4ms/step - loss: 1.9990 - val_loss: 1.8651
Epoch 8/10
13/13 [==============================] - 0s 4ms/step - loss: 2.1550 - val_loss: 1.3440
Epoch 9/10
13/13 [==============================] - 0s 5ms/step - loss: 1.6482 - val_loss: 1.5049
Epoch 10/10
13/13 [==============================] - 0s 4ms/step - loss: 1.9065 - val_loss: 1.3944


In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_kd.tflite") / 1024
print(f"Model size: {file_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpzqx4iil7/assets


INFO:tensorflow:Assets written to: /tmp/tmpzqx4iil7/assets


Model size: 6.11 KB


2026-05-20 21:02:31.876755: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:31.876811: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:02:31.876971: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpzqx4iil7
2026-05-20 21:02:31.878121: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:31.878138: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpzqx4iil7
2026-05-20 21:02:31.881335: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:31.919741: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpzqx4iil7
2026-05-20 21:02:31.930000: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 53029 m

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = student_model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)

print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 4ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        19
           1       1.00      0.19      0.32        21
           2       0.33      1.00      0.49        14

    accuracy                           0.33        54
   macro avg       0.44      0.40      0.27        54
weighted avg       0.47      0.33      0.25        54

[[ 0  0 19]
 [ 7  4 10]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# <-- (if needed) Enter your code here <--#

small_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    
    # Smaller hidden layers for reduced model size
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    
    tf.keras.layers.Dense(3, activation='softmax')
])

small_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = small_model.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)


converter = tf.lite.TFLiteConverter.from_keras_model(small_model)
tflite_model = converter.convert()

with open("small_model.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("small_model.tflite") / 1024
print(f"Small model size: {file_size_kb:.2f} KB")


quantize_and_evaluate(
    small_model,
    X_test_scaled,
    y_test_encoded,
    'float16',
    'small_model_float16.tflite'
)




Epoch 1/20
13/13 [==============================] - 0s 15ms/step - loss: 0.8128 - accuracy: 0.7172 - val_loss: 0.5451 - val_accuracy: 0.8000
Epoch 2/20
13/13 [==============================] - 0s 5ms/step - loss: 0.4004 - accuracy: 0.9293 - val_loss: 0.3181 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 5ms/step - loss: 0.1893 - accuracy: 0.9798 - val_loss: 0.1844 - val_accuracy: 0.9600
Epoch 4/20
13/13 [==============================] - 0s 5ms/step - loss: 0.0965 - accuracy: 0.9798 - val_loss: 0.1285 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 5ms/step - loss: 0.0564 - accuracy: 0.9899 - val_loss: 0.0915 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 5ms/step - loss: 0.0338 - accuracy: 0.9899 - val_loss: 0.0763 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 4ms/step - loss: 0.0220 - accuracy: 1.0000 - val_loss: 0.0622 - val_accuracy: 0.9600
Epoch 8/20
13/13 [=

INFO:tensorflow:Assets written to: /tmp/tmpbsg9z49o/assets
2026-05-20 21:02:34.288961: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:34.289052: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:02:34.289304: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpbsg9z49o
2026-05-20 21:02:34.290818: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:34.290841: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpbsg9z49o
2026-05-20 21:02:34.294609: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:34.343913: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpbsg9z49o
2026-05-20 21:02:34.357015: I tensorflow/cc/saved_model/loader.cc:316] SavedModel

Small model size: 6.12 KB
INFO:tensorflow:Assets written to: /tmp/tmpt0495emd/assets


INFO:tensorflow:Assets written to: /tmp/tmpt0495emd/assets



FLOAT16 TFLite model size: 5.02 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 21:02:34.909398: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:02:34.909455: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:02:34.909630: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpt0495emd
2026-05-20 21:02:34.910821: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:02:34.910839: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpt0495emd
2026-05-20 21:02:34.914308: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 21:02:34.958310: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpt0495emd
2026-05-20 21:02:34.969405: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 59774 m

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
